# Обучение модели классификации дорожного покрытия

Этот ноутбук предназначен для запуска на **Kaggle** с использованием GPU.

## Инструкция по запуску из локального репозитория

```bash
# 1. Скачать данные через DVC
make dvc-pull

# 2. Отправить ноутбук на Kaggle GPU
make kaggle-push NOTEBOOK=notebooks/kaggle/train_model.ipynb \
    TITLE="Road Surface Training" \
    WAIT=1

# 3. Или запустить локально
python scripts/train.py --config-name train_config
```

In [ ]:
# Проверка окружения Kaggle
import os

IS_KAGGLE = os.environ.get("KAGGLE_CONTAINER_NAME") is not None

if IS_KAGGLE:
    print("✓ Запуск на Kaggle")
    print(f"  GPU: {os.environ.get('NVIDIA_VISIBLE_DEVICES', 'N/A')}")
else:
    print("✓ Локальный запуск")

## Загрузка данных (для Kaggle)

На Kaggle данные скачиваются напрямую из Yandex Cloud S3.

In [ ]:
if IS_KAGGLE:
    import boto3
    from botocore.config import Config
    from pathlib import Path
    
    # Конфигурация S3 клиента
    BUCKET_NAME = os.environ.get("BUCKET_NAME", "road-surface-classification-storage-1")
    AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
    AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")
    
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        raise ValueError(
            "AWS credentials not found. "
            "Add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY to Kaggle Secrets."
        )
    
    # Создание S3 клиента
    s3_client = boto3.client(
        's3',
        endpoint_url='https://storage.yandexcloud.net',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        config=Config(signature_version='s3v4'),
    )
    
    # Пути
    DATA_ROOT = Path("/kaggle/working/data/raw")
    OUTPUT_DIR = Path("/kaggle/working")
    
    def download_from_s3(bucket: str, prefix: str, local_dir: Path) -> list:
        """Скачать файлы из S3 bucket."""
        local_dir.mkdir(parents=True, exist_ok=True)
        
        # Список объектов
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
        
        downloaded = []
        for page in pages:
            if 'Contents' not in page:
                continue
            for obj in page['Contents']:
                key = obj['Key']
                # Пропускаем "папки"
                if key.endswith('/'):
                    continue
                
                # Локальный путь
                relative_path = key[len(prefix):].lstrip('/')
                local_path = local_dir / relative_path
                local_path.parent.mkdir(parents=True, exist_ok=True)
                
                # Скачивание
                print(f"Downloading: {key} -> {local_path}")
                s3_client.download_file(bucket, key, str(local_path))
                downloaded.append(local_path)
        
        return downloaded
    
    # Скачивание данных
    print(f"\nDownloading data from s3://{BUCKET_NAME}/raw/...")
    downloaded_files = download_from_s3(BUCKET_NAME, "raw/", DATA_ROOT)
    print(f"Downloaded {len(downloaded_files)} files\n")
    
else:
    # Локально - данные уже есть через DVC
    DATA_ROOT = Path("../data/raw")
    OUTPUT_DIR = Path("../outputs")

In [ ]:
# Импорт библиотек
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchaudio
import numpy as np
from tqdm import tqdm
from rich import print as rprint

# Добавляем проект в path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.audio.data.preprocessing import AudioPreprocessor
from src.core.callbacks import DualLogger
from src.core.mlflow_config import setup_mlflow

## Конфигурация

In [ ]:
# Параметры
CONFIG = {
    "model_name": "resnet18",
    "num_classes": 5,
    "sample_rate": 16000,
    "n_mels": 128,
    "epochs": 30,
    "batch_size": 32,
    "learning_rate": 0.001,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

rprint(f"[cyan]Configuration:[/cyan] {CONFIG}")
rprint(f"[cyan]Data root:[/cyan] {DATA_ROOT}")
rprint(f"[cyan]Device:[/cyan] {CONFIG['device']}")

## Dataset

In [ ]:
class RoadSurfaceDataset(Dataset):
    """Dataset for road surface audio classification."""

    CLASS_NAMES = ["asphalt", "concrete", "gravel", "dirt", "snow"]

    def __init__(
        self,
        data_root: Path,
        sample_rate: int = 16000,
        duration: float = 5.0,
        augment: bool = False,
    ):
        self.data_root = data_root
        self.sample_rate = sample_rate
        self.duration = duration
        self.augment = augment
        
        self.samples = self._load_samples()
        self.preprocessor = AudioPreprocessor(
            sample_rate=sample_rate,
            n_mels=128,
        )

    def _load_samples(self) -> list[tuple[Path, int]]:
        """Load sample paths and labels."""
        samples = []
        
        for label, class_name in enumerate(self.CLASS_NAMES):
            class_dir = self.data_root / class_name
            if not class_dir.exists():
                continue
            
            for audio_file in class_dir.glob("*.wav"):
                samples.append((audio_file, label))
        
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        path, label = self.samples[idx]
        
        # Load audio
        waveform, sr = torchaudio.load(path)
        
        # Resample
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)
        
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Pad/truncate
        target_samples = int(self.duration * self.sample_rate)
        if waveform.shape[1] < target_samples:
            waveform = torch.nn.functional.pad(
                waveform, (0, target_samples - waveform.shape[1])
            )
        else:
            waveform = waveform[:, :target_samples]
        
        # Extract features
        features = self.preprocessor.extract_features(waveform.squeeze())
        
        return features, label

In [ ]:
# Создание dataloaders
from sklearn.model_selection import train_test_split

# Загрузка всех путей к файлам
all_samples = RoadSurfaceDataset._load_samples(
    type('Self', (), {'data_root': DATA_ROOT})()
)

# Разделение на train/val
train_indices, val_indices = train_test_split(
    list(range(len(all_samples))),
    test_size=0.2,
    random_state=42,
)

# Создание датасетов
train_dataset = RoadSurfaceDataset(DATA_ROOT, augment=True)
val_dataset = RoadSurfaceDataset(DATA_ROOT, augment=False)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=2,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=2,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

## Модель

In [ ]:
def create_model(model_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    """Create model from configuration."""
    from torchvision import models
    
    if "resnet" in model_name.lower():
        if model_name == "resnet18":
            backbone = models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        # Modify first layer for single-channel input
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Modify classifier
        in_features = backbone.fc.in_features
        backbone.fc = nn.Linear(in_features, num_classes)
        
        return backbone
    else:
        raise ValueError(f"Unsupported model: {model_name}")


model = create_model(CONFIG["model_name"], CONFIG["num_classes"])
model = model.to(CONFIG["device"])
rprint(f"[green]Model created:[/green] {CONFIG['model_name']}")

## Обучение

In [ ]:
# Loss и optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=0.0001,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
)

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(dataloader, desc="Training")
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Validating"):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    
    return avg_loss, accuracy

In [ ]:
# Setup логгеров
mlflow_config = setup_mlflow(
    tracking_uri=os.environ.get("MLFLOW_TRACKING_URI"),
    experiment_name="road-surface-kaggle",
)

with DualLogger(
    project="road-surface-classification",
    experiment_name="train_baseline",
    log_to_wandb=True,
    log_to_mlflow=True,
) as logger:
    
    logger.log_params(CONFIG)
    
    best_val_loss = float("inf")
    
    for epoch in range(CONFIG["epochs"]):
        # Train
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, CONFIG["device"]
        )
        
        # Validate
        val_loss, val_acc = validate(
            model, val_loader, criterion, CONFIG["device"]
        )
        
        # Update scheduler
        scheduler.step()
        
        # Log metrics
        logger.log_metrics({
            "train/loss": train_loss,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
        }, step=epoch)
        
        rprint(
            f"Epoch {epoch+1}/{CONFIG['epochs']}: "
            f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
            f"val_acc={val_acc:.1f}%"
        )
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            logger.log_model(model, artifact_name="best_model")

rprint("\n[green bold]Training complete![/green bold]")

## Сохранение модели

In [ ]:
# Сохранение для Kaggle
if IS_KAGGLE:
    output_path = OUTPUT_DIR / "model.pth"
    torch.save(model.state_dict(), output_path)
    print(f"Model saved to: {output_path}")
    
    # Список файлов для скачивания
    print("\nOutput files:")
    for f in OUTPUT_DIR.glob("*"):
        print(f"  {f}")